In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import time
from openai import OpenAI

class GrammarChecker:
    def __init__(self, api_key):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
        )

    def check_grammar(self, text):
        if pd.isna(text) or str(text).strip() == '':
            return ''

        try:
            completion = self.client.chat.completions.create(
                model="qwen-plus-latest",
                messages=[
                    {"role": "system", "content": """You are a grammar checker. Check the given sentence for grammar errors.

Rules:
- If there are grammar errors, respond with ONLY a brief description of the error(s)
- If there are NO errors, respond with exactly: "No errors"
- Be concise and specific about the errors found"""},
                    {"role": "user", "content": f'Check: "{text}"'}
                ],
                temperature=0,
                max_tokens=100
            )

            result = completion.choices[0].message.content.strip()
            return '' if 'no error' in result.lower() else result

        except Exception:
            time.sleep(2)
            return 'ERROR'

def process_csv(input_path, output_path, api_key):
    df = pd.read_csv(input_path)
    checker = GrammarChecker(api_key)

    # Find justification columns
    just_cols = [col for col in df.columns if col.startswith('justification_for_')]

    # Add grammar error columns
    for col in just_cols:
        feature = col.replace('justification_for_', '')
        df[f'grammar_error_{feature}'] = ''

    # Process each row
    for idx, row in df.iterrows():
        for just_col in just_cols:
            feature = just_col.replace('justification_for_', '')
            error = checker.check_grammar(row[just_col])
            df.at[idx, f'grammar_error_{feature}'] = error

        if idx % 10 == 0:
            print(f"Processed {idx}/{len(df)} rows")
            time.sleep(1)

    df.to_csv(output_path, index=False)
    print(f"Complete. Saved to {output_path}")

# Usage
API_KEY = os.environ["DASHSCOPE_API_KEY"]
INPUT_CSV = '/content/drive/MyDrive/VLM/final_sez.csv'
OUTPUT_CSV = '/content/drive/MyDrive/VLM/output_grammar_checked.csv'

process_csv(INPUT_CSV, OUTPUT_CSV, API_KEY)

In [ ]:
import os
import pandas as pd
import time
from openai import OpenAI

FEATURE_DEFINITIONS = {
    'gender': "Patient's biological sex, either male or female.",
    'occur_during_sleep': "Event of interest occurs while patient is asleep, as visualized in the video recording.",
    'head_turning': "Unnatural, forced, and sustained versive head turn. Versive movement: forced, sustained, involuntary turning of head and/or eyes to one side, resulting in an unnatural posture, typically contralateral to the ictal onset.",
    'blank_stare': "Staring with motor arrest. A sudden interruption of ongoing activities, accompanied by a loss of facial expression and an unfocused, vacant gaze. It is the hallmark of a typical absence seizure but can also occur in focal impaired consciousness seizures.",
    'close_eyes': "Eyes are bilaterally closed. Sustained and forceful closure of the eyelids during a convulsive event. This is more commonly observed in psychogenic non-epileptic seizures (PNES) than in epileptic seizures.",
    'eye_blinking': "Repetitive tonic contraction of the eyelids during a seizure. A series of repetitive tonic contractions of the eyelids during a seizure, typically at a frequency of 3 Hz or more.",
    'face_pulling': "Tonic contraction of the face. A sustained tonic or dystonic contraction of facial muscles, resulting in a grimace or distorted facial expression.",
    'face_twitching': "Brief, repetitive, involuntary clonic (rhythmic) or myoclonic (irregular, shock-like) jerks of the facial muscles.",
    'tonic': "A sustained increase in muscle tone, or stiffening, lasting seconds to minutes. It can be focal (affecting one limb) or generalized (affecting the entire body).",
    'clonic': "Sustained, rhythmic, bilateral jerking movements. In a generalized tonic-clonic seizure, the clonic phase follows the tonic phase.",
    'arm_straightening': "Sustained tonic or dystonic posturing characterized by extension of the arms, making them appear straight and rigid.",
    'arm_flexion': "Sustained tonic or dystonic posturing characterized by bending of the arms, typically at the elbows and/or shoulders.",
    'figure4': "An asymmetric limb posture where one arm is tonically extended while the other is flexed at the elbow, creating a shape resembling the number '4'. Also known as a 'fencer's posture.'",
    'oral_automatisms': "Coordinated, repetitive, semi-purposeful motor activity of the mouth, lips, and tongue, such as lip-smacking, chewing, or swallowing, occurring during a state of impaired consciousness.",
    'limb_automatisms': "Coordinated, repetitive, purposeless motor activity of the limbs, such as fumbling with clothing, picking at objects, or patting surfaces, occurring during a state of impaired consciousness.",
    'tremor': "Rhythmic, involuntary oscillatory movement of a body part.",
    'pelvic_thrusting': "Repetitive, rhythmic, anteroposterior (forward-and-backward) movements of the hips. It is a form of hyperkinetic motor activity.",
    'full_body_jerking': "Generalized jerking movements affecting the entire body including arms, legs, and torso.",
    'limb_movement_pattern': "Classification of upper limb movements: Thrashing/Flailing (significant thrashing and flailing, somewhat asynchronous and variable), Rhythmic Jerking (stereotyped, rhythmic clonic jerking), or Neither.",
    'arms_move_simultaneously': "Synchronous onset of bilateral motor activity, indicating engagement of both cerebral hemispheres from the start.",
    'intensity_evolution': "Evolution of motor activity intensity: Crescendo pattern (gradual and rhythmic increase in intensity, amplitude, or frequency) or Stable pattern (constant intensity for significant duration).",
    'verbal_responsiveness': "The ability to comprehend and provide a relevant verbal response to a question or command during an ictal event. Its absence is a key indicator of impaired consciousness. Can be 'yes', 'no', or 'N/A' if not tested.",
    'ictal_vocalization': "Non-speech sounds (e.g., groan, cry, shriek) or verbal automatisms (repetitive words/phrases) produced during the seizure (ictus)."
}

class AlignmentChecker:
    def __init__(self, api_key):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
        )

    def check_alignment(self, feature_name, actual_answer, justification):
        if pd.isna(justification) or str(justification).strip() == '':
            return 'invalid'

        feature_definition = FEATURE_DEFINITIONS.get(feature_name, f"Feature: {feature_name}")

        system_prompt = f"""You are analyzing whether a justification text supports a given answer for an epilepsy feature.

FEATURE DEFINITION:
{feature_definition}

TASK:
- You will be given an actual answer ('yes', 'no', or 'N/A') and a justification text
- Determine if the justification supports the given answer
- Focus on whether the reasoning in the justification logically leads to the stated answer

RESPONSE FORMAT:
Respond with ONLY one of these options:
- "aligned" - if the justification supports the given answer
- "misaligned" - if the justification contradicts or does not support the given answer
- "unclear" - if the justification is ambiguous or insufficient to determine alignment"""

        user_prompt = f"""FEATURE: {feature_name}
ACTUAL ANSWER: {actual_answer}
JUSTIFICATION: "{justification}"

Does the justification support the actual answer given?"""

        try:
            completion = self.client.chat.completions.create(
                model="qwen-plus-latest",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0,
                max_tokens=50
            )

            result = completion.choices[0].message.content.strip().lower()

            if 'aligned' in result and 'misaligned' not in result:
                return 'aligned'
            elif 'misaligned' in result:
                return 'misaligned'
            elif 'unclear' in result:
                return 'unclear'
            else:
                return 'unclear'

        except Exception:
            time.sleep(2)
            return 'ERROR'

def process_csv(input_path, output_path, api_key):
    df = pd.read_csv(input_path)
    checker = AlignmentChecker(api_key)

    # Find feature-justification pairs
    just_cols = [col for col in df.columns if col.startswith('justification_for_')]
    feature_pairs = []

    for just_col in just_cols:
        feature_name = just_col.replace('justification_for_', '')
        possible_cols = [feature_name, f"answer_for_{feature_name}", f"{feature_name}_answer"]

        feature_col = None
        for possible_col in possible_cols:
            if possible_col in df.columns:
                feature_col = possible_col
                break

        if feature_col:
            feature_pairs.append((feature_name, feature_col, just_col))

    # Add alignment check columns
    for feature_name, _, _ in feature_pairs:
        df[f'alignment_check_{feature_name}'] = ''

    # Process each row
    for idx, row in df.iterrows():
        for feature_name, feature_col, just_col in feature_pairs:
            actual_answer = row[feature_col]
            justification = row[just_col]

            alignment = checker.check_alignment(feature_name, actual_answer, justification)
            df.at[idx, f'alignment_check_{feature_name}'] = alignment

        if idx % 10 == 0:
            print(f"Processed {idx}/{len(df)} rows")
            time.sleep(1)

    df.to_csv(output_path, index=False)
    print(f"Complete. Saved to {output_path}")

# Usage
API_KEY = os.environ["DASHSCOPE_API_KEY"]
INPUT_CSV = '/content/drive/MyDrive/VLM/final_sez.csv'
OUTPUT_CSV = '/content/drive/MyDrive/VLM/output_alignment_checked.csv'

process_csv(INPUT_CSV, OUTPUT_CSV, API_KEY)